# BirdCLEF 2026 — Perch ONNX Inference

**Before running on Kaggle:**
1. Upload the 6 files from `logs/perch_memory/` as a new Kaggle dataset
2. Add that dataset + `rishikeshjani/perch-onnx-for-birdclef-2026` to this notebook
3. Update `MODEL_DATASET_SLUG` below to match your dataset slug

In [ ]:
import subprocess, importlib

# Try to import onnxruntime — if already installed, skip pip install
try:
    import onnxruntime
    print(f'onnxruntime already available: {onnxruntime.__version__}')
except ImportError:
    _whl = '/kaggle/input/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'
    subprocess.run(['pip', 'install', '-q', '--no-deps', _whl], check=True)
    print('onnxruntime installed from local whl')

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import onnxruntime as ort
from pathlib import Path

# Kaggle mounts datasets at /kaggle/input/{dataset-name}/ (just the slug, no owner prefix)
MODEL_DATASET_SLUG = 'birdclef-perch-head-v1'

# Auto-detect competition directory
for _cand in [
    Path('/kaggle/input/birdclef-2026'),
    Path('/kaggle/input/competitions/birdclef-2026'),
]:
    if (_cand / 'sample_submission.csv').exists():
        COMP_DIR = _cand
        break
else:
    COMP_DIR = Path('/kaggle/input/birdclef-2026')

# Auto-detect ONNX path
for _cand in [
    Path('/kaggle/input/perch-onnx-for-birdclef-2026/perch_v2.onnx'),
    Path('/kaggle/input/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx'),
    Path('/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx'),
]:
    if _cand.exists():
        ONNX_PATH = _cand
        break
else:
    ONNX_PATH = Path('/kaggle/input/perch-onnx-for-birdclef-2026/perch_v2.onnx')

# Auto-detect model dir
for _cand in [
    Path(f'/kaggle/input/{MODEL_DATASET_SLUG}'),
    Path(f'/kaggle/input/moritzblacker/{MODEL_DATASET_SLUG}'),
    Path(f'/kaggle/input/datasets/moritzblacker/{MODEL_DATASET_SLUG}'),
]:
    if _cand.exists():
        MODEL_DIR = _cand
        break
else:
    MODEL_DIR = Path(f'/kaggle/input/{MODEL_DATASET_SLUG}')

SR           = 32_000
CLIP_SEC     = 5.0
CLIP_SAMPLES = int(SR * CLIP_SEC)

print(f'COMP_DIR:  {COMP_DIR}  (exists={COMP_DIR.exists()})')
print(f'ONNX:      {ONNX_PATH}  (exists={ONNX_PATH.exists()})')
print(f'MODEL_DIR: {MODEL_DIR}  (exists={MODEL_DIR.exists()})')

test_dir = COMP_DIR / 'test_soundscapes'
print(f'test_soundscapes: exists={test_dir.exists()}')
if test_dir.exists():
    files = list(test_dir.iterdir())
    print(f'  {len(files)} files  first 5: {[f.name for f in files[:5]]}')

In [ ]:
# ── Load Perch ONNX ────────────────────────────────────────────────────────
so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess     = ort.InferenceSession(str(ONNX_PATH), sess_options=so, providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name

dummy     = np.zeros((1, CLIP_SAMPLES), dtype=np.float32)
outs      = sess.run(None, {inp_name: dummy})
emb_idx   = next(i for i, o in enumerate(outs) if o.ndim == 2 and o.shape[-1] == 1536)
logit_idx = next(i for i, o in enumerate(outs) if o.ndim == 2 and o.shape[-1] > 5000)
print(f'Embedding: output[{emb_idx}] {outs[emb_idx].shape}')
print(f'Logits:    output[{logit_idx}] {outs[logit_idx].shape}')

In [ ]:
# ── Load trained head ──────────────────────────────────────────────────────
# Rebuild architecture from spec + load weights only — avoids Keras 2/3 compat issues

model_dir = MODEL_DIR
print('Model files at:', model_dir)
for f in sorted(model_dir.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(model_dir)}  ({f.stat().st_size/1e6:.1f} MB)')

with open(model_dir / 'best_model_info.json') as f:
    info = json.load(f)
spec = info['spec']
perch_weight = float(spec.get('perch_weight', 0.2))
print(f'perch_weight={perch_weight}  (training AUC={info["auc"]:.5f})')
print(f'spec: n_blocks={spec["n_blocks"]} hidden={spec["hidden_dim"]} proj={spec["proj_dim"]}')

def build_head_ref(spec, emb_dim, num_classes):
    n_blocks     = int(spec.get('n_blocks',      2))
    hidden_dim   = int(spec.get('hidden_dim',    1024))
    proj_dim     = int(spec.get('proj_dim',      512))
    drop_block   = float(spec.get('dropout_block', 0.3))
    drop_final   = float(spec.get('dropout_final', 0.4))

    inp = tf.keras.layers.Input(shape=(emb_dim,))
    x   = tf.keras.layers.BatchNormalization()(inp)
    x   = tf.keras.layers.Dense(hidden_dim)(x)
    x   = tf.keras.layers.LayerNormalization()(x)
    for _ in range(n_blocks):
        h = tf.keras.layers.Dense(hidden_dim)(x)
        h = tf.keras.layers.LayerNormalization()(h)
        h = tf.keras.layers.Activation('gelu')(h)
        h = tf.keras.layers.Dropout(drop_block)(h)
        h = tf.keras.layers.Dense(hidden_dim)(h)
        x = tf.keras.layers.Add()([x, h])
        x = tf.keras.layers.LayerNormalization()(x)
    x   = tf.keras.layers.Dense(proj_dim, activation='gelu')(x)
    x   = tf.keras.layers.Dropout(drop_final)(x)
    out = tf.keras.layers.Dense(num_classes, activation='sigmoid')(x)
    return tf.keras.Model(inp, out)

# Load species cols first so we know num_classes
with open(model_dir / 'species_cols.json') as f:
    species_cols_tmp = json.load(f)

head = build_head_ref(spec, emb_dim=1536, num_classes=len(species_cols_tmp))
print(f'Architecture built: {head.input_shape} → {head.output_shape}  params={head.count_params():,}')

weights_path = model_dir / 'best_head.weights.h5'
if not weights_path.exists():
    import zipfile
    with zipfile.ZipFile(str(model_dir / 'best_head.keras'), 'r') as z:
        z.extract('model.weights.h5', '/tmp/keras_weights/')
    weights_path = Path('/tmp/keras_weights/model.weights.h5')

head.load_weights(str(weights_path))
print('Weights loaded OK')

In [ ]:
# ── Load species mapping ────────────────────────────────────────────────────
with open(model_dir / 'species_cols.json') as f:
    species_cols = json.load(f)
with open(model_dir / 'mapping_meta.json') as f:
    meta = json.load(f)
with open(model_dir / 'proxy_map.json') as f:
    proxy_map = {int(k): v for k, v in json.load(f).items()}

NO_LABEL   = int(meta['NO_LABEL'])
n_species  = int(meta['n_species'])
bc_indices = np.load(str(model_dir / 'bc_indices.npy'))

MAPPED_MASK   = bc_indices != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = bc_indices[MAPPED_MASK].astype(np.int32)
print(f'Mapped: {MAPPED_MASK.sum()}/{n_species}  Genus proxy: {len(proxy_map)}')

sample_sub  = pd.read_csv(COMP_DIR / 'sample_submission.csv')
sub_species = [c for c in sample_sub.columns if c != 'row_id']
assert sub_species == species_cols, 'Species column mismatch!'
print(f'Species columns OK: {len(sub_species)}')

In [ ]:
# ── Inference helpers ───────────────────────────────────────────────────────
def embed_batch(wavs):
    outs = sess.run(None, {inp_name: wavs})
    return outs[emb_idx].astype(np.float32), outs[logit_idx].astype(np.float32)

def map_logits(logits):
    B   = logits.shape[0]
    out = np.full((B, n_species), logits.mean(axis=1, keepdims=True), dtype=np.float32)
    out[:, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
    for sp_idx, bc_idxs in proxy_map.items():
        out[:, sp_idx] = logits[:, bc_idxs].mean(axis=1)
    return out

def predict_file(fpath, batch_size=8):
    wav_full, _ = librosa.load(str(fpath), sr=SR, mono=True)
    name        = Path(fpath).stem
    segments, end_secs = [], []
    for i, start in enumerate(range(0, len(wav_full), CLIP_SAMPLES)):
        seg = wav_full[start : start + CLIP_SAMPLES]
        if len(seg) < CLIP_SAMPLES:
            seg = np.pad(seg, (0, CLIP_SAMPLES - len(seg)))
        segments.append(seg.astype(np.float32))
        end_secs.append((i + 1) * int(CLIP_SEC))
    rows = []
    for b in range(0, len(segments), batch_size):
        batch       = np.stack(segments[b : b + batch_size])
        emb, lgts   = embed_batch(batch)
        head_probs  = head.predict(emb, verbose=0)
        perch_probs = 1.0 / (1.0 + np.exp(-map_logits(lgts)))
        blended     = perch_weight * perch_probs + (1.0 - perch_weight) * head_probs
        for j, preds in enumerate(blended):
            row = {'row_id': f'{name}_{end_secs[b + j]}'}
            for col, p in zip(species_cols, preds):
                row[col] = float(np.clip(p, 0.0, 1.0))
            rows.append(row)
    return rows

print('Helpers ready.')

In [ ]:
# ── Run inference (stem-based — same approach as working notebooks) ─────────
sample_sub  = pd.read_csv(COMP_DIR / 'sample_submission.csv')
sub_species = [c for c in sample_sub.columns if c != 'row_id']
sample_sub['stem']    = sample_sub['row_id'].str.rsplit('_', n=1).str[0]
sample_sub['end_sec'] = sample_sub['row_id'].str.rsplit('_', n=1).str[1].astype(int)
unique_stems = sample_sub['stem'].unique()
print(f'Test files: {len(unique_stems)} | rows: {len(sample_sub)}')

test_dir = COMP_DIR / 'test_soundscapes'
all_rows = []

t0 = time.time()
for i, stem in enumerate(unique_stems):
    fpath = test_dir / f'{stem}.ogg'
    if not fpath.is_file():
        print(f'  Missing: {fpath}')
        continue
    y_full, _ = librosa.load(str(fpath), sr=SR, mono=True)
    rows_for_file = sample_sub[sample_sub['stem'] == stem]

    segs, row_ids = [], []
    for ridx, r in rows_for_file.iterrows():
        end_samp   = int(r['end_sec'] * SR)
        start_samp = max(0, end_samp - CLIP_SAMPLES)
        seg = y_full[start_samp:end_samp]
        if len(seg) < CLIP_SAMPLES:
            seg = np.pad(seg, (0, CLIP_SAMPLES - len(seg)))
        segs.append(seg.astype(np.float32))
        row_ids.append(r['row_id'])

    emb, lgts   = embed_batch(np.stack(segs))
    head_probs  = head.predict(emb, verbose=0)
    perch_probs = 1.0 / (1.0 + np.exp(-map_logits(lgts)))
    blended     = perch_weight * perch_probs + (1.0 - perch_weight) * head_probs

    for j, row_id in enumerate(row_ids):
        row = {'row_id': row_id}
        for col, p in zip(species_cols, blended[j]):
            row[col] = float(np.clip(p, 0.0, 1.0))
        all_rows.append(row)

    if (i + 1) % 25 == 0:
        print(f'  [{i+1}/{len(unique_stems)}]  {(i+1)/(time.time()-t0):.1f} files/s')

print(f'Done in {time.time()-t0:.1f}s')
print(f'Prediction rows: {len(all_rows)}')

In [ ]:
# ── Save submission ─────────────────────────────────────────────────────────
print(f'Prediction rows generated: {len(all_rows)}')

if all_rows:
    sub = sample_sub[['row_id']].merge(pd.DataFrame(all_rows), on='row_id', how='left')
    sub[sub_species] = sub[sub_species].fillna(0.0).clip(0.0, 1.0)
else:
    # No test audio available yet (development phase) — submit zeros as placeholder
    print('WARNING: No test soundscapes found. Submitting placeholder zeros.')
    sub = sample_sub[['row_id'] + sub_species].copy()
    sub[sub_species] = 0.0

sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'submission.csv saved  shape={sub.shape}  columns={list(sub.columns[:4])} ...')
sub.head(3)